# StreamSense — TFX Training Pipeline + Kubeflow (self-contained, all fixes baked in)

Runs the **TFX training pipeline end-to-end** on Colab and compiles it to a Kubeflow v2 spec.
Everything that used to break is fixed here, so just **Run all** top to bottom:

- `uv` + a pinned lock install TFX on a managed **Python 3.10** (no condacolab, no restart, no crash)
- `uv venv --seed` + explicit `dill` give TFX the pip/deps it needs to ship user-code wheels
- the training module is patched in-session: `vocab_filename`, GZIP reader, OOV buckets, a
  `tf.Example` serving signature, Evaluator config, and a DNS-style KFP pipeline name

> No GPU needed (CPU is correct for MovieLens). No cell needs manual editing — just Run all.

## 1. Install uv + clone the repo (no restart)

In [ ]:
!pip install -q uv
!git clone https://github.com/techykajal/streamsense-ott-recsys.git
%cd streamsense-ott-recsys
!ls requirements-tfx-lock.txt

## 2. Create a Python 3.10 venv and install TFX from the lock — FAST
`uv` fetches a standalone CPython 3.10 and installs the pinned deps. `--seed` adds pip/setuptools/
wheel (TFX ships user-code via pip); `dill` is an apache-beam transitive dep TFX needs for v1 components.

In [ ]:
!uv venv --seed --python 3.10 /content/tfxenv
!uv pip install --python /content/tfxenv/bin/python -r requirements-tfx-lock.txt
!uv pip install --python /content/tfxenv/bin/python "dill==0.3.1.1"
!/content/tfxenv/bin/python -c "import tfx, tensorflow as tf; print('TFX', tfx.__version__, '| TF', tf.__version__, 'OK')" 

## 3. Apply the source fixes in-session
Overwrites `src/trainer_module.py` and `src/tfx_pipeline.py` with the corrected versions so this
notebook runs end-to-end regardless of what's on GitHub. (Same content you can also `git push`.)

In [ ]:
import base64, pathlib
pathlib.Path("src/trainer_module.py").write_bytes(base64.b64decode(
"IiIiCnRyYWluZXJfbW9kdWxlLnB5IOKAlCBtb2R1bGVfZmlsZSBzaGFyZWQgYnkgVEZYIFRyYW5zZm9ybSArIFRyYWluZXIuCgpwcmVwcm9jZXNzaW5nX2ZuIDogZmVhdHVyZSBlbmdpbmVlcmluZyBpbnNpZGUgdGhlIFRyYW5zZm9ybSBncmFwaC4KcnVuX2ZuICAgICAgICAgICA6IHRyYWlucyBhIHNtYWxsIEtlcmFzIHJhbmtpbmcgbW9kZWwgYW5kIHNhdmVzIGl0IFdJVEggYSBzZXJ2aW5nCiAgICAgICAgICAgICAgICAgICBzaWduYXR1cmUgdGhhdCBwYXJzZXMgcmF3IHRmLkV4YW1wbGUgLT4gYXBwbGllcyB0aGUgdGYuVHJhbnNmb3JtCiAgICAgICAgICAgICAgICAgICBsYXllciAtPiBydW5zIHRoZSBtb2RlbC4gVGhhdCBsZXRzIFRGIFNlcnZpbmcgYW5kIHRoZSBURlggRXZhbHVhdG9yCiAgICAgICAgICAgICAgICAgICBmZWVkIHJhdyBleGFtcGxlcyBkaXJlY3RseSAobm8gY2xpZW50LXNpZGUgcHJlcHJvY2Vzc2luZykuClRoaXMgaXMgdGhlIFRGLXNpZGUgcmFua2VyOyB0aGUgUHlUb3JjaCByYW5rZXIgKHJhbmtpbmdfdG9yY2gucHkpIGlzIHRoZSBUcml0b24vVG9yY2hTZXJ2ZSBvbmUuCiIiIgppbXBvcnQgdGVuc29yZmxvdyBhcyB0ZgppbXBvcnQgdGVuc29yZmxvd190cmFuc2Zvcm0gYXMgdGZ0CgpVU0VSLCBNT1ZJRSwgU0VHLCBMQUJFTCA9ICJ1c2VyX2lkIiwgIm1vdmllX2lkIiwgInNlZ21lbnQiLCAibGFiZWwiCgoKZGVmIHByZXByb2Nlc3NpbmdfZm4oaW5wdXRzKToKICAgIG91dCA9IHt9CiAgICAjIG51bV9vb3ZfYnVja2V0cz0xIC0+IElEcyB1bnNlZW4gaW4gdGhlIHRyYWluIHNwbGl0IG1hcCB0byBhIFZBTElEIG91dC1vZi12b2NhYiBpbmRleAogICAgIyAoPSB2b2NhYl9zaXplKSBpbnN0ZWFkIG9mIHRoZSBkZWZhdWx0IC0xLCBzbyB0aGUgZW1iZWRkaW5nIGxvb2t1cCBuZXZlciBnZXRzIGEKICAgICMgbmVnYXRpdmUgaW5kZXggKHRoYXQgd2FzIHRoZSAiaW5kaWNlcyA9IC0xIGlzIG5vdCBpbiBbMCwgTikiIGVycm9yKS4KICAgIG91dFtVU0VSXSA9IHRmdC5jb21wdXRlX2FuZF9hcHBseV92b2NhYnVsYXJ5KAogICAgICAgIGlucHV0c1tVU0VSXSwgdG9wX2s9NTBfMDAwLCBudW1fb292X2J1Y2tldHM9MSwgdm9jYWJfZmlsZW5hbWU9InVzZXJfdm9jYWIiKQogICAgb3V0W01PVklFXSA9IHRmdC5jb21wdXRlX2FuZF9hcHBseV92b2NhYnVsYXJ5KAogICAgICAgIGlucHV0c1tNT1ZJRV0sIHRvcF9rPTUwXzAwMCwgbnVtX29vdl9idWNrZXRzPTEsIHZvY2FiX2ZpbGVuYW1lPSJtb3ZpZV92b2NhYiIpCiAgICBvdXRbU0VHXSA9IHRmLmNhc3QoaW5wdXRzW1NFR10sIHRmLmludDY0KQogICAgb3V0W0xBQkVMXSA9IHRmLmNhc3QoaW5wdXRzW0xBQkVMXSwgdGYuZmxvYXQzMikKICAgIHJldHVybiBvdXQKCgpkZWYgX2lucHV0X2ZuKGZpbGVzLCB0Zl90cmFuc2Zvcm1fb3V0cHV0LCBiYXRjaD0xMDI0KToKICAgIHNwZWMgPSB0Zl90cmFuc2Zvcm1fb3V0cHV0LnRyYW5zZm9ybWVkX2ZlYXR1cmVfc3BlYygpLmNvcHkoKQogICAgZHMgPSB0Zi5kYXRhLmV4cGVyaW1lbnRhbC5tYWtlX2JhdGNoZWRfZmVhdHVyZXNfZGF0YXNldCgKICAgICAgICBmaWxlcywgYmF0Y2gsIHNwZWMsIGxhYmVsX2tleT1MQUJFTCwgc2h1ZmZsZT1UcnVlLCBudW1fZXBvY2hzPU5vbmUsCiAgICAgICAgcmVhZGVyX2FyZ3M9WyJHWklQIl0pICAgICAgICAgICMgVEZYIFRyYW5zZm9ybSB3cml0ZXMgR1pJUC1jb21wcmVzc2VkIFRGUmVjb3JkcwogICAgcmV0dXJuIGRzCgoKZGVmIF9tb2RlbChuX3VzZXIsIG5fbW92aWUsIG5fc2VnLCBkaW09MzIpOgogICAgdSA9IHRmLmtlcmFzLklucHV0KHNoYXBlPSgxLCksIG5hbWU9VVNFUiwgZHR5cGU9dGYuaW50NjQpCiAgICBtID0gdGYua2VyYXMuSW5wdXQoc2hhcGU9KDEsKSwgbmFtZT1NT1ZJRSwgZHR5cGU9dGYuaW50NjQpCiAgICBzID0gdGYua2VyYXMuSW5wdXQoc2hhcGU9KDEsKSwgbmFtZT1TRUcsIGR0eXBlPXRmLmludDY0KQogICAgIyArMSBsZWF2ZXMgcm9vbSBmb3IgdGhlIHNpbmdsZSBPT1YgYnVja2V0IGluZGV4ICg9IHZvY2FiX3NpemUpLgogICAgdWUgPSB0Zi5rZXJhcy5sYXllcnMuRW1iZWRkaW5nKG5fdXNlciArIDEsIGRpbSkodSkKICAgIG1lID0gdGYua2VyYXMubGF5ZXJzLkVtYmVkZGluZyhuX21vdmllICsgMSwgZGltKShtKQogICAgc2UgPSB0Zi5rZXJhcy5sYXllcnMuRW1iZWRkaW5nKG5fc2VnICsgMSwgZGltKShzKQogICAgeCA9IHRmLmtlcmFzLmxheWVycy5Db25jYXRlbmF0ZSgpKFt0Zi5rZXJhcy5sYXllcnMuRmxhdHRlbigpKHQpIGZvciB0IGluICh1ZSwgbWUsIHNlKV0pCiAgICB4ID0gdGYua2VyYXMubGF5ZXJzLkRlbnNlKDEyOCwgYWN0aXZhdGlvbj0icmVsdSIpKHgpCiAgICB4ID0gdGYua2VyYXMubGF5ZXJzLkRlbnNlKDY0LCBhY3RpdmF0aW9uPSJyZWx1IikoeCkKICAgIG91dCA9IHRmLmtlcmFzLmxheWVycy5EZW5zZSgxLCBhY3RpdmF0aW9uPSJzaWdtb2lkIikoeCkKICAgIG1vZGVsID0gdGYua2VyYXMuTW9kZWwoW3UsIG0sIHNdLCBvdXQpCiAgICBtb2RlbC5jb21waWxlKG9wdGltaXplcj0iYWRhbSIsIGxvc3M9ImJpbmFyeV9jcm9zc2VudHJvcHkiLAogICAgICAgICAgICAgICAgICBtZXRyaWNzPVt0Zi5rZXJhcy5tZXRyaWNzLkFVQyhuYW1lPSJhdWMiKV0pCiAgICByZXR1cm4gbW9kZWwKCgpkZWYgX3NlcnZpbmdfc2lnbmF0dXJlKG1vZGVsLCB0Zl90cmFuc2Zvcm1fb3V0cHV0KToKICAgICIiInNlcnZpbmdfZGVmYXVsdDogcmF3IHNlcmlhbGl6ZWQgdGYuRXhhbXBsZSAtPiB0ZnQgdHJhbnNmb3JtIC0+IG1vZGVsIHByZWRpY3Rpb24uIiIiCiAgICBtb2RlbC50ZnRfbGF5ZXIgPSB0Zl90cmFuc2Zvcm1fb3V0cHV0LnRyYW5zZm9ybV9mZWF0dXJlc19sYXllcigpCgogICAgQHRmLmZ1bmN0aW9uKGlucHV0X3NpZ25hdHVyZT1bCiAgICAgICAgdGYuVGVuc29yU3BlYyhzaGFwZT1bTm9uZV0sIGR0eXBlPXRmLnN0cmluZywgbmFtZT0iZXhhbXBsZXMiKV0pCiAgICBkZWYgc2VydmVfZm4oc2VyaWFsaXplZCk6CiAgICAgICAgZmVhdHVyZV9zcGVjID0gdGZfdHJhbnNmb3JtX291dHB1dC5yYXdfZmVhdHVyZV9zcGVjKCkKICAgICAgICBmZWF0dXJlX3NwZWMucG9wKExBQkVMLCBOb25lKSAgICAgICAgICAgICAgICAgIyBsYWJlbCBpcyBhYnNlbnQgYXQgc2VydmluZyB0aW1lCiAgICAgICAgcGFyc2VkID0gdGYuaW8ucGFyc2VfZXhhbXBsZShzZXJpYWxpemVkLCBmZWF0dXJlX3NwZWMpCiAgICAgICAgdHJhbnNmb3JtZWQgPSBtb2RlbC50ZnRfbGF5ZXIocGFyc2VkKSAgICAgICAgICMgYXBwbGllcyB2b2NhYiBtYXBwaW5nIGV0Yy4KICAgICAgICBwcmVkID0gbW9kZWwoe1VTRVI6IHRyYW5zZm9ybWVkW1VTRVJdLAogICAgICAgICAgICAgICAgICAgICAgTU9WSUU6IHRyYW5zZm9ybWVkW01PVklFXSwKICAgICAgICAgICAgICAgICAgICAgIFNFRzogdHJhbnNmb3JtZWRbU0VHXX0pCiAgICAgICAgcmV0dXJuIHsicHJlZGljdGlvbiI6IHByZWR9CiAgICByZXR1cm4gc2VydmVfZm4KCgpkZWYgcnVuX2ZuKGZuX2FyZ3MpOgogICAgdGZ0byA9IHRmdC5URlRyYW5zZm9ybU91dHB1dChmbl9hcmdzLnRyYW5zZm9ybV9vdXRwdXQpCiAgICB0cmFpbiA9IF9pbnB1dF9mbihmbl9hcmdzLnRyYWluX2ZpbGVzLCB0ZnRvKQogICAgdmFsID0gX2lucHV0X2ZuKGZuX2FyZ3MuZXZhbF9maWxlcywgdGZ0bykKICAgIG5fdXNlciA9IHRmdG8udm9jYWJ1bGFyeV9zaXplX2J5X25hbWUoInVzZXJfdm9jYWIiKQogICAgbl9tb3ZpZSA9IHRmdG8udm9jYWJ1bGFyeV9zaXplX2J5X25hbWUoIm1vdmllX3ZvY2FiIikKICAgIG1vZGVsID0gX21vZGVsKG5fdXNlciwgbl9tb3ZpZSwgbl9zZWc9NjQpCiAgICBtb2RlbC5maXQodHJhaW4sIHN0ZXBzX3Blcl9lcG9jaD1mbl9hcmdzLnRyYWluX3N0ZXBzLAogICAgICAgICAgICAgIHZhbGlkYXRpb25fZGF0YT12YWwsIHZhbGlkYXRpb25fc3RlcHM9Zm5fYXJncy5ldmFsX3N0ZXBzLCBlcG9jaHM9MSkKICAgIHNpZ25hdHVyZXMgPSB7InNlcnZpbmdfZGVmYXVsdCI6IF9zZXJ2aW5nX3NpZ25hdHVyZShtb2RlbCwgdGZ0byl9CiAgICBtb2RlbC5zYXZlKGZuX2FyZ3Muc2VydmluZ19tb2RlbF9kaXIsIHNhdmVfZm9ybWF0PSJ0ZiIsIHNpZ25hdHVyZXM9c2lnbmF0dXJlcykK"))
pathlib.Path("src/tfx_pipeline.py").write_bytes(base64.b64decode(
"IiIiCnRmeF9waXBlbGluZS5weSDigJQgVEZYIHBpcGVsaW5lIGZvciB0aGUgcmFua2luZyBtb2RlbCArIEt1YmVmbG93IGNvbXBpbGUuCgogICAgcHl0aG9uIHNyYy90ZnhfcGlwZWxpbmUucHkgICAgICAgICAgICAgICAgIyBydW4gbG9jYWxseSAoTG9jYWxEYWdSdW5uZXIpCiAgICBweXRob24gc3JjL3RmeF9waXBlbGluZS5weSAtLWNvbXBpbGUta2ZwICAjIGVtaXQgb3R0X3BpcGVsaW5lLnlhbWwgKEt1YmVmbG93IFBpcGVsaW5lcyB2MiBJUikKCkNvbXBvbmVudHM6IEV4YW1wbGVHZW4gLT4gU3RhdGlzdGljc0dlbiAtPiBTY2hlbWFHZW4gLT4gRXhhbXBsZVZhbGlkYXRvcgogICAgICAgICAgICAtPiBUcmFuc2Zvcm0gLT4gVHJhaW5lciAtPiBFdmFsdWF0b3IgLT4gUHVzaGVyCgpUaGUgVHJhaW5lciBtb2R1bGUgKHRyYWluZXJfbW9kdWxlLnB5LCBzaWJsaW5nIGZpbGUpIGJ1aWxkcyBhIHNtYWxsIEtlcmFzIHJhbmtpbmcgbW9kZWwuClRoaXMgaXMgdGhlICJkYXRhIHBpcGVsaW5lIGVuZ2luZWVyaW5nIiArICJleHBlcmltZW50YXRpb24gKEV2YWx1YXRvcikiIHN0b3J5IGZyb20gdGhlIEpELAphbmQgdGhlIFNBTUUgZ3JhcGggY29tcGlsZXMgdG8gS3ViZWZsb3cgd2l0aCBhIG9uZS1saW5lIHJ1bm5lciBzd2FwLgoiIiIKaW1wb3J0IG9zCmltcG9ydCBzeXMKCmZyb20gdGZ4IGltcG9ydCB2MSBhcyB0ZngKZnJvbSB0ZngucHJvdG8gaW1wb3J0IGV4YW1wbGVfZ2VuX3BiMgoKUElQRUxJTkVfTkFNRSA9ICJvdHRfcmFua2luZyIKREFUQV9ST09UID0gb3MucGF0aC5hYnNwYXRoKCJkYXRhL3Byb2Nlc3NlZCIpICAgICAgICAgICMgaG9sZHMgaW50ZXJhY3Rpb25zLnRmcmVjb3JkCk1PRFVMRSA9IG9zLnBhdGguYWJzcGF0aCgic3JjL3RyYWluZXJfbW9kdWxlLnB5IikKUElQRUxJTkVfUk9PVCA9IG9zLnBhdGguYWJzcGF0aChmInRmeC97UElQRUxJTkVfTkFNRX0iKQpNRVRBREFUQSA9IG9zLnBhdGguam9pbihQSVBFTElORV9ST09ULCAibWV0YWRhdGEuZGIiKQpTRVJWSU5HID0gb3MucGF0aC5hYnNwYXRoKCJhcnRpZmFjdHMvcmFua2luZ190ZiIpCgoKZGVmIGJ1aWxkX3BpcGVsaW5lKHBpcGVsaW5lX3Jvb3QsIG1ldGFkYXRhX2Nvbm5lY3Rpb249Tm9uZSwKICAgICAgICAgICAgICAgICAgIHBpcGVsaW5lX25hbWU9UElQRUxJTkVfTkFNRSkgLT4gdGZ4LmRzbC5QaXBlbGluZToKICAgIGV4YW1wbGVfZ2VuID0gdGZ4LmNvbXBvbmVudHMuSW1wb3J0RXhhbXBsZUdlbigKICAgICAgICBpbnB1dF9iYXNlPURBVEFfUk9PVCwKICAgICAgICBpbnB1dF9jb25maWc9ZXhhbXBsZV9nZW5fcGIyLklucHV0KHNwbGl0cz1bCiAgICAgICAgICAgIGV4YW1wbGVfZ2VuX3BiMi5JbnB1dC5TcGxpdChuYW1lPSJzaW5nbGUiLCBwYXR0ZXJuPSJpbnRlcmFjdGlvbnMudGZyZWNvcmQiKSwKICAgICAgICBdKSwKICAgICAgICBvdXRwdXRfY29uZmlnPWV4YW1wbGVfZ2VuX3BiMi5PdXRwdXQoCiAgICAgICAgICAgIHNwbGl0X2NvbmZpZz1leGFtcGxlX2dlbl9wYjIuU3BsaXRDb25maWcoc3BsaXRzPVsKICAgICAgICAgICAgICAgIGV4YW1wbGVfZ2VuX3BiMi5TcGxpdENvbmZpZy5TcGxpdChuYW1lPSJ0cmFpbiIsIGhhc2hfYnVja2V0cz04KSwKICAgICAgICAgICAgICAgIGV4YW1wbGVfZ2VuX3BiMi5TcGxpdENvbmZpZy5TcGxpdChuYW1lPSJldmFsIiwgaGFzaF9idWNrZXRzPTIpLAogICAgICAgICAgICBdKSkpCgogICAgc3RhdHMgPSB0ZnguY29tcG9uZW50cy5TdGF0aXN0aWNzR2VuKGV4YW1wbGVzPWV4YW1wbGVfZ2VuLm91dHB1dHNbImV4YW1wbGVzIl0pCiAgICBzY2hlbWEgPSB0ZnguY29tcG9uZW50cy5TY2hlbWFHZW4oc3RhdGlzdGljcz1zdGF0cy5vdXRwdXRzWyJzdGF0aXN0aWNzIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgaW5mZXJfZmVhdHVyZV9zaGFwZT1UcnVlKQogICAgdmFsaWRhdG9yID0gdGZ4LmNvbXBvbmVudHMuRXhhbXBsZVZhbGlkYXRvcihzdGF0aXN0aWNzPXN0YXRzLm91dHB1dHNbInN0YXRpc3RpY3MiXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc2NoZW1hPXNjaGVtYS5vdXRwdXRzWyJzY2hlbWEiXSkKICAgIHRyYW5zZm9ybSA9IHRmeC5jb21wb25lbnRzLlRyYW5zZm9ybSgKICAgICAgICBleGFtcGxlcz1leGFtcGxlX2dlbi5vdXRwdXRzWyJleGFtcGxlcyJdLAogICAgICAgIHNjaGVtYT1zY2hlbWEub3V0cHV0c1sic2NoZW1hIl0sCiAgICAgICAgbW9kdWxlX2ZpbGU9TU9EVUxFKQogICAgdHJhaW5lciA9IHRmeC5jb21wb25lbnRzLlRyYWluZXIoCiAgICAgICAgbW9kdWxlX2ZpbGU9TU9EVUxFLAogICAgICAgIGV4YW1wbGVzPXRyYW5zZm9ybS5vdXRwdXRzWyJ0cmFuc2Zvcm1lZF9leGFtcGxlcyJdLAogICAgICAgIHRyYW5zZm9ybV9ncmFwaD10cmFuc2Zvcm0ub3V0cHV0c1sidHJhbnNmb3JtX2dyYXBoIl0sCiAgICAgICAgc2NoZW1hPXNjaGVtYS5vdXRwdXRzWyJzY2hlbWEiXSwKICAgICAgICB0cmFpbl9hcmdzPXRmeC5wcm90by5UcmFpbkFyZ3MobnVtX3N0ZXBzPTIwMDApLAogICAgICAgIGV2YWxfYXJncz10ZngucHJvdG8uRXZhbEFyZ3MobnVtX3N0ZXBzPTUwMCkpCiAgICAjIEV2YWx1YXRvciAoVEZNQSk6IGZlZWQgUkFXIGV4YW1wbGVzOyB0aGUgbW9kZWwncyBzZXJ2aW5nX2RlZmF1bHQgc2lnbmF0dXJlIHBhcnNlcwogICAgIyB0Zi5FeGFtcGxlIC0+IGFwcGxpZXMgdGhlIHRmLlRyYW5zZm9ybSBsYXllciAtPiBwcmVkaWN0cywgc28gbm8gY2xpZW50IHByZXByb2Nlc3NpbmcuCiAgICBpbXBvcnQgdGVuc29yZmxvd19tb2RlbF9hbmFseXNpcyBhcyB0Zm1hCiAgICBldmFsX2NvbmZpZyA9IHRmbWEuRXZhbENvbmZpZygKICAgICAgICBtb2RlbF9zcGVjcz1bdGZtYS5Nb2RlbFNwZWMoc2lnbmF0dXJlX25hbWU9InNlcnZpbmdfZGVmYXVsdCIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGxhYmVsX2tleT0ibGFiZWwiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBwcmVkaWN0aW9uX2tleT0icHJlZGljdGlvbiIpXSwKICAgICAgICBzbGljaW5nX3NwZWNzPVt0Zm1hLlNsaWNpbmdTcGVjKCldLCAgICAgICAgICAgICAgICAgIyBvdmVyYWxsIHNsaWNlCiAgICAgICAgbWV0cmljc19zcGVjcz1bdGZtYS5NZXRyaWNzU3BlYyhtZXRyaWNzPVsKICAgICAgICAgICAgdGZtYS5NZXRyaWNDb25maWcoY2xhc3NfbmFtZT0iRXhhbXBsZUNvdW50IiksCiAgICAgICAgICAgIHRmbWEuTWV0cmljQ29uZmlnKGNsYXNzX25hbWU9IkFVQyIpLAogICAgICAgICAgICB0Zm1hLk1ldHJpY0NvbmZpZyhjbGFzc19uYW1lPSJCaW5hcnlBY2N1cmFjeSIpLAogICAgICAgIF0pXSkKICAgIGV2YWx1YXRvciA9IHRmeC5jb21wb25lbnRzLkV2YWx1YXRvcigKICAgICAgICBleGFtcGxlcz1leGFtcGxlX2dlbi5vdXRwdXRzWyJleGFtcGxlcyJdLAogICAgICAgIG1vZGVsPXRyYWluZXIub3V0cHV0c1sibW9kZWwiXSwKICAgICAgICBldmFsX2NvbmZpZz1ldmFsX2NvbmZpZykKICAgIHB1c2hlciA9IHRmeC5jb21wb25lbnRzLlB1c2hlcigKICAgICAgICBtb2RlbD10cmFpbmVyLm91dHB1dHNbIm1vZGVsIl0sCiAgICAgICAgcHVzaF9kZXN0aW5hdGlvbj10ZngucHJvdG8uUHVzaERlc3RpbmF0aW9uKAogICAgICAgICAgICBmaWxlc3lzdGVtPXRmeC5wcm90by5QdXNoRGVzdGluYXRpb24uRmlsZXN5c3RlbShiYXNlX2RpcmVjdG9yeT1TRVJWSU5HKSkpCgogICAgY29tcG9uZW50cyA9IFtleGFtcGxlX2dlbiwgc3RhdHMsIHNjaGVtYSwgdmFsaWRhdG9yLCB0cmFuc2Zvcm0sCiAgICAgICAgICAgICAgICAgIHRyYWluZXIsIGV2YWx1YXRvciwgcHVzaGVyXQogICAgcmV0dXJuIHRmeC5kc2wuUGlwZWxpbmUoCiAgICAgICAgcGlwZWxpbmVfbmFtZT1waXBlbGluZV9uYW1lLAogICAgICAgIHBpcGVsaW5lX3Jvb3Q9cGlwZWxpbmVfcm9vdCwKICAgICAgICBjb21wb25lbnRzPWNvbXBvbmVudHMsCiAgICAgICAgbWV0YWRhdGFfY29ubmVjdGlvbl9jb25maWc9bWV0YWRhdGFfY29ubmVjdGlvbikKCgpkZWYgcnVuX2xvY2FsKCk6CiAgICBtZCA9IHRmeC5vcmNoZXN0cmF0aW9uLm1ldGFkYXRhLnNxbGl0ZV9tZXRhZGF0YV9jb25uZWN0aW9uX2NvbmZpZyhNRVRBREFUQSkKICAgIHAgPSBidWlsZF9waXBlbGluZShQSVBFTElORV9ST09ULCBtZCkKICAgIHRmeC5vcmNoZXN0cmF0aW9uLkxvY2FsRGFnUnVubmVyKCkucnVuKHApCgoKZGVmIGNvbXBpbGVfa2ZwKG91dD0ib3R0X3BpcGVsaW5lLnlhbWwiKToKICAgICIiIkNvbXBpbGUgdGhlIGlkZW50aWNhbCBEQUcgdG8gS3ViZWZsb3cgUGlwZWxpbmVzIHYyIElSIChydW5zIG9uIFZlcnRleCBBSSAvIEdLRSBLRlApLgoKICAgIFRoZSBjb21waWxlciBzdGFnZXMgdGhlIHVzZXItY29kZSB3aGVlbHMgdW5kZXIgcGlwZWxpbmVfcm9vdCBhbmQgY2hlY2tzIGl0IGV4aXN0cywgc28gd2UKICAgIG5lZWQgYSByZWFsLCByZWFjaGFibGUgcm9vdC4gRm9yIGFuIGFjdHVhbCBWZXJ0ZXggcnVuIHNldCBLRlBfUElQRUxJTkVfUk9PVCB0byB5b3VyIGJ1Y2tldDoKICAgICAgICBLRlBfUElQRUxJTkVfUk9PVD1nczovLzx5b3VyLWJ1Y2tldD4vb3R0X3JhbmtpbmcgcHl0aG9uIHNyYy90ZnhfcGlwZWxpbmUucHkgLS1jb21waWxlLWtmcAogICAgV2l0aCBubyBlbnYgdmFyIHdlIGRlZmF1bHQgdG8gYSBMT0NBTCByb290IHNvIHRoZSBZQU1MIGNvbXBpbGVzIGFueXdoZXJlIChubyBHQ1MgbmVlZGVkKS4KICAgICIiIgogICAgZnJvbSB0Zngub3JjaGVzdHJhdGlvbi5rdWJlZmxvdy52MiBpbXBvcnQga3ViZWZsb3dfdjJfZGFnX3J1bm5lciBhcyBrZnBfcnVubmVyCiAgICBrZnBfcm9vdCA9IG9zLmVudmlyb24uZ2V0KCJLRlBfUElQRUxJTkVfUk9PVCIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG9zLnBhdGguYWJzcGF0aChmInRmeF9rZnAve1BJUEVMSU5FX05BTUV9IikpCiAgICBpZiBub3Qga2ZwX3Jvb3Quc3RhcnRzd2l0aCgiZ3M6Ly8iKToKICAgICAgICBvcy5tYWtlZGlycyhrZnBfcm9vdCwgZXhpc3Rfb2s9VHJ1ZSkgICAgICMgbG9jYWwgcm9vdCBtdXN0IGV4aXN0IGZvciB3aGVlbCBzdGFnaW5nCiAgICBydW5uZXIgPSBrZnBfcnVubmVyLkt1YmVmbG93VjJEYWdSdW5uZXIoCiAgICAgICAgY29uZmlnPWtmcF9ydW5uZXIuS3ViZWZsb3dWMkRhZ1J1bm5lckNvbmZpZygKICAgICAgICAgICAgZGVmYXVsdF9pbWFnZT0iZ2NyLmlvL3RmeC1vc3MtcHVibGljL3RmeDoxLjE1LjEiKSwKICAgICAgICBvdXRwdXRfZmlsZW5hbWU9b3V0KQogICAgIyBLRlAgdjIgcGlwZWxpbmUgbmFtZXMgbXVzdCBiZSBETlMtc3R5bGU6IFthLXowLTldW2EtejAtOS1dKiAobm8gdW5kZXJzY29yZXMpLgogICAgcnVubmVyLnJ1bihidWlsZF9waXBlbGluZShwaXBlbGluZV9yb290PWtmcF9yb290LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICBwaXBlbGluZV9uYW1lPVBJUEVMSU5FX05BTUUucmVwbGFjZSgiXyIsICItIikpKQogICAgcHJpbnQoZiJDb21waWxlZCBLdWJlZmxvdyB2MiBwaXBlbGluZSDihpIge291dH0gIChwaXBlbGluZV9yb290PXtrZnBfcm9vdH0pIikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgaWYgIi0tY29tcGlsZS1rZnAiIGluIHN5cy5hcmd2OgogICAgICAgIGNvbXBpbGVfa2ZwKCkKICAgIGVsc2U6CiAgICAgICAgcnVuX2xvY2FsKCkK"))
print("Patched src/trainer_module.py and src/tfx_pipeline.py")


## 4. Prepare the pipeline's training data
Builds `data/processed/interactions.tfrecord` (the pipeline's input).

In [ ]:
!/content/tfxenv/bin/python src/features.py

## 5. Run the TFX pipeline END-TO-END  ⭐
ExampleGen → StatisticsGen → SchemaGen → ExampleValidator → Transform → Trainer → Evaluator → Pusher.

In [ ]:
!rm -rf tfx/ott_ranking tfx_kfp
!/content/tfxenv/bin/python src/tfx_pipeline.py
print("\n=== Component artifacts produced by the pipeline ===")
!find tfx/ott_ranking -maxdepth 2 -type d 2>/dev/null | sort
print("\n=== Pushed (deployable) model ===")
!ls -R artifacts/ranking_tf 2>/dev/null | head -30 || echo "(Pusher dest = artifacts/ranking_tf)" 

## 6. Compile the pipeline for Kubeflow (Vertex AI / GKE)

In [ ]:
!/content/tfxenv/bin/python src/tfx_pipeline.py --compile-kfp
print("=== ott_pipeline.yaml (first 45 lines) ===")
!head -45 ott_pipeline.yaml

## 7. Done — TFX pipeline executed end-to-end ✅

Full DAG on real x86 Linux + Python 3.10, plus the Kubeflow v2 spec.

- **Train (this notebook):** TFX validates → transforms → trains → evaluates → **pushes** a model,
  reproducibly; compiled to a Kubeflow spec for Vertex AI / GKE.
- **Registry:** the pushed model is versioned (POC: `models/`; prod: MLflow / Vertex / S3).
- **Serve (other notebook):** TF Serving / TorchServe / Triton **load** the pushed model.

**Save to your repo:** File -> Save a copy in GitHub -> `techykajal/streamsense-ott-recsys`,
branch `main`, path `notebooks/StreamSense_Colab_TFX.ipynb`.